# Gikomba Market Stall Performance Analysis

This notebook uses core Python to evaluate Wanjiku's two-week fresh-produce stall performance, growth, customer timing, and stock priorities.

In [ ]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Overall sales performance
I calculate the total, average, maximum, and minimum sales, then use the positions of the extrema to identify the day name and week number.

In [ ]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest_sales = max(daily_sales)
lowest_sales = min(daily_sales)
highest_position = daily_sales.index(highest_sales)
lowest_position = daily_sales.index(lowest_sales)

print(f"Total sales: KES {total_sales:,.0f}")
print(f"Average daily sales: KES {average_sales:,.2f}")
print(f"Highest sales: KES {highest_sales:,.0f} on {days[highest_position]} (week {highest_position // 7 + 1})")
print(f"Lowest sales: KES {lowest_sales:,.0f} on {days[lowest_position]} (week {lowest_position // 7 + 1})")

## Task 2: Weekly totals and growth
The function uses list slicing to return each seven-day total. I then compare week 2 with week 1 to calculate growth.

In [ ]:
def week_totals(sales):
    """Return the sales totals for week 1 and week 2."""
    return sum(sales[:7]), sum(sales[7:])

week_1_total, week_2_total = week_totals(daily_sales)
growth_percentage = (week_2_total - week_1_total) / week_1_total * 100
print(f"Week 1 total: KES {week_1_total:,.0f}")
print(f"Week 2 total: KES {week_2_total:,.0f}")
print(f"Week-on-week growth: {growth_percentage:.2f}%")

## Task 3: Classify daily performance
Each day is classified against the overall average. A day is Strong when it is more than 20% above average and Weak when it is more than 20% below average.

In [ ]:
def classify_day(amount, average):
    """Return Strong, Normal, or Weak based on a 20% average threshold."""
    if amount > average * 1.2:
        return "Strong"
    if amount < average * 0.8:
        return "Weak"
    return "Normal"

classification_counts = {"Strong": 0, "Normal": 0, "Weak": 0}
daily_classifications = []
for amount in daily_sales:
    category = classify_day(amount, average_sales)
    daily_classifications.append(category)
    classification_counts[category] += 1

print(f"Classification counts: {classification_counts}")

## Task 4: Weekday versus weekend sales
I group sales by their day names, calculate the two averages, and express the weekend advantage as a percentage of the weekday average.

In [ ]:
weekday_sales = [amount for amount, day in zip(daily_sales, days) if day in ["Mon", "Tue", "Wed", "Thu", "Fri"]]
weekend_sales = [amount for amount, day in zip(daily_sales, days) if day in ["Sat", "Sun"]]
weekday_average = sum(weekday_sales) / len(weekday_sales)
weekend_average = sum(weekend_sales) / len(weekend_sales)
weekend_difference = (weekend_average - weekday_average) / weekday_average * 100

print(f"Weekday average: KES {weekday_average:,.2f}")
print(f"Weekend average: KES {weekend_average:,.2f}")
print(f"Weekend difference: {weekend_difference:.2f}% higher than weekdays")

## Task 5: Restocking and gross margins
The reorder function returns products whose quantity is at or below the reorder level. I also calculate the gross margin percentage for every product.

In [ ]:
def products_to_reorder(inventory):
    """Return product names at or below their reorder levels."""
    return [name for name, details in inventory.items() if details["qty"] <= details["reorder"]]

reorder_list = products_to_reorder(stock)
print(f"Products to reorder: {reorder_list}")
for product, details in stock.items():
    margin = (details["price"] - details["cost"]) / details["price"] * 100
    print(f"{product} gross margin: {margin:.2f}%")

## Task 6: Stall Performance Report
This report brings the main sales, growth, customer timing, and stock results together using f-strings and KES thousands separators.

In [ ]:
print("STALL PERFORMANCE REPORT")
print("========================")
print(f"14-day sales: KES {total_sales:,.0f}")
print(f"Average daily sales: KES {average_sales:,.2f}")
print(f"Week 1 / Week 2: KES {week_1_total:,.0f} / KES {week_2_total:,.0f}")
print(f"Growth: {growth_percentage:.2f}%")
print(f"Best day: {days[highest_position]}, week {highest_position // 7 + 1} (KES {highest_sales:,.0f})")
print(f"Weakest day: {days[lowest_position]}, week {lowest_position // 7 + 1} (KES {lowest_sales:,.0f})")
print(f"Weekend average: KES {weekend_average:,.2f}")
print(f"Restock: {', '.join(reorder_list)}")

## Task 7: Recommendations for Wanjiku
Wanjiku should prioritize Saturday and Sunday stock and staffing because the weekend average was KES 4,900, which was 82.50% higher than the weekday average. She should restock Tomatoes and Sukuma immediately because their quantities, 18 kg and 9 kg, were at or below their reorder levels of 25 kg and 15 kg. Week 2 sales reached KES 24,400, up 10.66% from week 1, so she should maintain the practices that produced this growth while tracking sales for another two weeks. She should review Tuesday promotions or purchasing because the weakest result was KES 1,950 on a Tuesday, and six of the 14 days were classified as Weak. Sukuma deserves attention in the product mix because it needs restocking and has a healthy 40.00% gross margin.